# Indexing Practice Notebook
#### INFO 376 — Search & Recommenders
 This notebook introduces indexing concepts step by step:
 - Tokenization
 - Stopword removal
 - Stemming
 - Inverted Index construction
 - Querying the index
 - Practice exercises


# our corpus

In [1]:
# Book titles as IDs and one-liner descriptions as content
documents = {
    "HarryPotter": "A young wizard discovers his magical heritage and battles dark forces.",
    "LordOfTheRings": "A hobbit carries a powerful ring on a perilous journey to destroy it.",
    "PrideAndPrejudice": "A classic novel exploring love, class, and misunderstandings in England.",
    "GreatGatsby": "A mysterious millionaire pursues the American dream in the Jazz Age.",
    "1984": "A dystopian society is controlled by surveillance, propaganda, and Big Brother.",
    "ToKillAMockingbird": "A lawyer defends an innocent black man accused of a crime in the South."
}

## Tokenization
- **Sentence tokenization** splits text into sentences.  
- **Word tokenization** splits sentences into words.  

Both are useful, but for indexing, we focus on **words**.


In [2]:
!pip install nltk --upgrade pip

  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached regex-2025.9.18-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
Using cached regex-2025.9.18-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (798 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nltk]1/2 [nltk]


In [3]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Pick one document for demonstration
sample_text = documents["HarryPotter"]

print("Sentence Tokenization:")
print(nltk.sent_tokenize(sample_text)) # Splits into sentences

print("\nWord Tokenization:")
print(nltk.word_tokenize(sample_text)) # Splits into words


[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Sentence Tokenization:
['A young wizard discovers his magical heritage and battles dark forces.']

Word Tokenization:
['A', 'young', 'wizard', 'discovers', 'his', 'magical', 'heritage', 'and', 'battles', 'dark', 'forces', '.']


#### Observation:
Sentence tokenization is useful for tasks like summarization.
Word tokenization is essential for indexing.

In [4]:
for doc_id, text in documents.items():
    tokens = nltk.word_tokenize(text)
    print(doc_id, ":", tokens)


HarryPotter : ['A', 'young', 'wizard', 'discovers', 'his', 'magical', 'heritage', 'and', 'battles', 'dark', 'forces', '.']
LordOfTheRings : ['A', 'hobbit', 'carries', 'a', 'powerful', 'ring', 'on', 'a', 'perilous', 'journey', 'to', 'destroy', 'it', '.']
PrideAndPrejudice : ['A', 'classic', 'novel', 'exploring', 'love', ',', 'class', ',', 'and', 'misunderstandings', 'in', 'England', '.']
GreatGatsby : ['A', 'mysterious', 'millionaire', 'pursues', 'the', 'American', 'dream', 'in', 'the', 'Jazz', 'Age', '.']
1984 : ['A', 'dystopian', 'society', 'is', 'controlled', 'by', 'surveillance', ',', 'propaganda', ',', 'and', 'Big', 'Brother', '.']
ToKillAMockingbird : ['A', 'lawyer', 'defends', 'an', 'innocent', 'black', 'man', 'accused', 'of', 'a', 'crime', 'in', 'the', 'South', '.']


## Stopword Removal
Stopwords are very common words (like "the", "is", "and") that add little meaning.
Removing them reduces index size and focuses on informative words.


In [5]:
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
print("Tokens after removing stopwords:\n")

for doc_id, text in documents.items():
    tokens = nltk.word_tokenize(text.lower()) # Lowercase + tokenize
    filtered = [w for w in tokens if w.isalpha() and w not in stop_words]
    print(f"{doc_id}: {filtered}")


Tokens after removing stopwords:

HarryPotter: ['young', 'wizard', 'discovers', 'magical', 'heritage', 'battles', 'dark', 'forces']
LordOfTheRings: ['hobbit', 'carries', 'powerful', 'ring', 'perilous', 'journey', 'destroy']
PrideAndPrejudice: ['classic', 'novel', 'exploring', 'love', 'class', 'misunderstandings', 'england']
GreatGatsby: ['mysterious', 'millionaire', 'pursues', 'american', 'dream', 'jazz', 'age']
1984: ['dystopian', 'society', 'controlled', 'surveillance', 'propaganda', 'big', 'brother']
ToKillAMockingbird: ['lawyer', 'defends', 'innocent', 'black', 'man', 'accused', 'crime', 'south']


[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Observation:

Common words like "the", "and", "in" are removed.

Focus remains on meaningful terms.

## Stemming & Lemmatization
Stemming reduces words to their root form.  
Examples:  
- "running", "runs" → "run"  
This way, variations of the same word are grouped together.


In [6]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download('wordnet')

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print(" Stemming vs Lemmatization:\n")
for doc_id, text in documents.items():
    tokens = nltk.word_tokenize(text.lower())
    filtered = [w for w in tokens if w.isalpha() and w not in stop_words]
    stemmed = [stemmer.stem(w) for w in filtered]
    lemmatized = [lemmatizer.lemmatize(w) for w in filtered]
    print(f"{doc_id}:")
    print(" Stemmed    →", stemmed)
    print(" Lemmatized →", lemmatized, "\n")


 Stemming vs Lemmatization:



[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


HarryPotter:
 Stemmed    → ['young', 'wizard', 'discov', 'magic', 'heritag', 'battl', 'dark', 'forc']
 Lemmatized → ['young', 'wizard', 'discovers', 'magical', 'heritage', 'battle', 'dark', 'force'] 

LordOfTheRings:
 Stemmed    → ['hobbit', 'carri', 'power', 'ring', 'peril', 'journey', 'destroy']
 Lemmatized → ['hobbit', 'carry', 'powerful', 'ring', 'perilous', 'journey', 'destroy'] 

PrideAndPrejudice:
 Stemmed    → ['classic', 'novel', 'explor', 'love', 'class', 'misunderstand', 'england']
 Lemmatized → ['classic', 'novel', 'exploring', 'love', 'class', 'misunderstanding', 'england'] 

GreatGatsby:
 Stemmed    → ['mysteri', 'millionair', 'pursu', 'american', 'dream', 'jazz', 'age']
 Lemmatized → ['mysterious', 'millionaire', 'pursues', 'american', 'dream', 'jazz', 'age'] 

1984:
 Stemmed    → ['dystopian', 'societi', 'control', 'surveil', 'propaganda', 'big', 'brother']
 Lemmatized → ['dystopian', 'society', 'controlled', 'surveillance', 'propaganda', 'big', 'brother'] 

ToKillAMock

##### Observation:

Stemming chops words to crude roots (battles → battl).

Lemmatization finds dictionary roots (battles → battle).

## Inverted Index
An inverted index maps terms → documents where they appear.

For example:  
- "wizard" → [HarryPotter]  
- "ring" → [LordOfTheRings]  

We will build a dictionary where each term points to a list of doc IDs.


In [7]:
# Inverted Index: term → documents containing it
inverted_index = {}

for doc_id, text in documents.items():
    tokens = nltk.word_tokenize(text.lower())
    filtered = [w for w in tokens if w.isalpha() and w not in stop_words]
    stemmed = [stemmer.stem(w) for w in filtered]
    
    for token in stemmed:
        if token not in inverted_index:
            inverted_index[token] = set()
        inverted_index[token].add(doc_id)

# Convert sets to sorted lists for readability
inverted_index = {term: sorted(list(docs)) for term, docs in inverted_index.items()}

print(" Sample of inverted index:")
for term in list(inverted_index.keys())[:10]:
    print(f"{term}: {inverted_index[term]}")


 Sample of inverted index:
young: ['HarryPotter']
wizard: ['HarryPotter']
discov: ['HarryPotter']
magic: ['HarryPotter']
heritag: ['HarryPotter']
battl: ['HarryPotter']
dark: ['HarryPotter']
forc: ['HarryPotter']
hobbit: ['LordOfTheRings']
carri: ['LordOfTheRings']


#### Observation:

The inverted index is the core of search engines.

It maps words directly to relevant documents → fast lookup.

## Querying the Index
To answer a query, we just look up the term in the inverted index.
We will stem the query term so that it matches our index.


In [8]:
def search(query, use_lemma=False):
    # Tokenize query
    tokens = nltk.word_tokenize(query.lower())
    print("\nRaw Tokens:", tokens)
    
    # Remove stopwords & non-alphabetic tokens
    filtered = [w for w in tokens if w.isalpha() and w not in stop_words]
    print("After Stopword Removal:", filtered)
    
    if use_lemma:
        # Apply Lemmatization
        processed = [lemmatizer.lemmatize(w) for w in filtered]
        print("After Lemmatization:", processed)
    else:
        # Apply Stemming
        processed = [stemmer.stem(w) for w in filtered]
        print("After Stemming:", processed)
    
    # Lookup postings
    print(processed)
    results = [inverted_index.get(t, []) for t in processed]
    return set.intersection(*map(set, results)) if results else []

print(" Query Results with Stemming:")
#print("Query: 'wizard' →", search("wizard"))
print("Query: 'force' →", search("force"))
print("Query: 'dream' →", search("dreaming"))
#print("Query: 'lawyer' →", search("lawyer"))

print("\n Query Results with Lemmatization:")
#print("Query: 'wizard' →", search("wizard", use_lemma=True))
print("Query: 'force' →", search("force", use_lemma=True))   
print("Query: 'dream' →", search("dreaming", use_lemma=True))
#print("Query: 'lawyer' →", search("lawyers", use_lemma=True))


 Query Results with Stemming:

Raw Tokens: ['force']
After Stopword Removal: ['force']
After Stemming: ['forc']
['forc']
Query: 'force' → {'HarryPotter'}

Raw Tokens: ['dreaming']
After Stopword Removal: ['dreaming']
After Stemming: ['dream']
['dream']
Query: 'dream' → {'GreatGatsby'}

 Query Results with Lemmatization:

Raw Tokens: ['force']
After Stopword Removal: ['force']
After Lemmatization: ['force']
['force']
Query: 'force' → set()

Raw Tokens: ['dreaming']
After Stopword Removal: ['dreaming']
After Lemmatization: ['dreaming']
['dreaming']
Query: 'dream' → set()


In [9]:
def search(query, use_lemma=False, match_all=True):
    # Tokenize
    tokens = nltk.word_tokenize(query.lower())
    print("\nRaw Tokens:", tokens)
    
    # Stopword removal
    filtered = [w for w in tokens if w.isalpha() and w not in stop_words]
    print("After Stopword Removal:", filtered)
    
    # Process tokens
    if use_lemma:
        processed = [lemmatizer.lemmatize(w) for w in filtered]
        print("After Lemmatization:", processed)
    else:
        processed = [stemmer.stem(w) for w in filtered]
        print("After Stemming:", processed)
    
    # Lookup postings for each term
    postings = [set(inverted_index.get(t, [])) for t in processed]
    
    if not postings:
        return set()
    
    # Intersection (AND) → all terms must appear
    if match_all:
        return set.intersection(*postings)
    # Union (OR) → any term can appear
    else:
        return set.union(*postings)


In [10]:
print(" Sentence Query with Stemming (AND match):")
print(search("A young wizard battles dark forces"))

print("\n Sentence Query with Lemmatization (AND match):")
print(search("A young wizard battles dark forces", use_lemma=True))

print("\n Sentence Query (OR match):")
print(search("wizard ring dream", match_all=False))


 Sentence Query with Stemming (AND match):

Raw Tokens: ['a', 'young', 'wizard', 'battles', 'dark', 'forces']
After Stopword Removal: ['young', 'wizard', 'battles', 'dark', 'forces']
After Stemming: ['young', 'wizard', 'battl', 'dark', 'forc']
{'HarryPotter'}

 Sentence Query with Lemmatization (AND match):

Raw Tokens: ['a', 'young', 'wizard', 'battles', 'dark', 'forces']
After Stopword Removal: ['young', 'wizard', 'battles', 'dark', 'forces']
After Lemmatization: ['young', 'wizard', 'battle', 'dark', 'force']
set()

 Sentence Query (OR match):

Raw Tokens: ['wizard', 'ring', 'dream']
After Stopword Removal: ['wizard', 'ring', 'dream']
After Stemming: ['wizard', 'ring', 'dream']
{'HarryPotter', 'GreatGatsby', 'LordOfTheRings'}
